In [2]:
# mT5 TRAINING - JUST THE BASICS

# Step 1: Install what we need
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "datasets", "torch", "sacrebleu"])

# Step 2: Import libraries
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from datasets import Dataset

# Step 3: Load your CSV files (UPDATE THESE PATHS!)
train_h = pd.read_csv("/Users/komalbadgujar/Desktop/codemixtranslation/hinglish_train.csv")
train_s = pd.read_csv("/Users/komalbadgujar/Desktop/codemixtranslation/spanglish_train.csv")

# Step 4: Combine and prepare data
train = pd.concat([train_h, train_s])
train["input"] = "translate: " + train["source"].astype(str)
train["output"] = train["target"].astype(str)
train = train[["input", "output"]]

# Step 5: Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

# Step 6: Tokenize
def tokenize(examples):
    inputs = tokenizer(examples["input"], max_length=128, truncation=True)
    targets = tokenizer(examples["output"], max_length=128, truncation=True)
    inputs["labels"] = targets["input_ids"]
    return inputs

train_ds = Dataset.from_pandas(train)
train_ds = train_ds.map(tokenize, batched=True, remove_columns=["input", "output"])

# Step 7: Training settings
args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    save_strategy="no",
    logging_steps=100,
    report_to=[],
    use_cpu=True,  # Force CPU to avoid memory issues
)

# Step 8: Train
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()

# Step 9: Save
model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")
print("Done! Model saved to ./my_model")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/opt/anaconda3/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

/var/folders/bp/5qpnmnwj34bb5s53ljplk1cc0000gn/T/ipykernel_36482/491906917.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Step,Training Loss
100,20.670900
200,16.134000
300,12.764400
400,9.095700
500,7.324000
600,6.682000
700,5.535300
800,5.137100
900,4.947700
1000,4.818300


Done! Model saved to ./my_model


In [4]:
# EVAL CELL 
import pandas as pd, torch, json, re, os
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu

# Paths (update if your val/test files live elsewhere)
HING_VAL = "/Users/komalbadgujar/Desktop/codemixtranslation/hinglish_val.csv"
SPAN_VAL = "/Users/komalbadgujar/Desktop/codemixtranslation/spanglish_val.csv"
HING_TST = "/Users/komalbadgujar/Desktop/codemixtranslation/hinglish_test.csv"
SPAN_TST = "/Users/komalbadgujar/Desktop/codemixtranslation/spanglish_test.csv"

# Load your just-saved model
model_dir = "./my_model"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

def _norm(x: str) -> str:
    return re.sub(r"\s+"," ", x.strip().lower())

def eval_split(csv_path: str, out_name: str, batch_size: int = 16):
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["source","target"]).reset_index(drop=True)
    inputs = ("translate: " + df["source"].astype(str)).tolist()
    refs   = df["target"].astype(str).tolist()

    preds = []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            out = model.generate(**{k:v for k,v in enc.items()},
                                 max_length=96, num_beams=4)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

    # Metrics
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    em = 100.0 * sum(_norm(p)==_norm(r) for p,r in zip(preds, refs)) / max(1,len(refs))

    # Save artifacts
    os.makedirs("./eval_outputs", exist_ok=True)
    pd.DataFrame({"source": df["source"], "prediction": preds, "reference": refs}) \
      .to_csv(f"./eval_outputs/preds_{out_name}.csv", index=False)
    with open(f"./eval_outputs/metrics_{out_name}.json", "w") as f:
        json.dump({"bleu": bleu, "chrf": chrf, "exact_match": em}, f, indent=2)

    print(f"[{out_name}] BLEU={bleu:.2f}  chrF={chrf:.2f}  EM={em:.2f}")
    return bleu, chrf, em

print("— VALIDATION —")
eval_split(HING_VAL, "hinglish_val")
eval_split(SPAN_VAL, "spanglish_val")

print("\n— TEST —")
eval_split(HING_TST, "hinglish_test")
eval_split(SPAN_TST, "spanglish_test")


The tokenizer you are loading from './my_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


— VALIDATION —
[hinglish_val] BLEU=4.06  chrF=15.51  EM=0.00
[spanglish_val] BLEU=17.72  chrF=30.48  EM=0.00

— TEST —
[hinglish_test] BLEU=3.24  chrF=14.06  EM=0.00
[spanglish_test] BLEU=19.38  chrF=32.36  EM=0.00


(19.380119434251085, 32.36452969592939, 0.0)